# Snippet from Cookbook.md


In [ ]:
"""Illustrative smoke tests against a real CompitumRouter.

The repo's actual smoke tests live at tests/examples/test_examples_smoke.py
(marked pytest.mark.examples, excluded by pytest.ini's default addopts -- run
with `pytest -q -m examples tests/examples/test_examples_smoke.py`). This
notebook shows the *pattern* for writing your own, using real field names.
"""
import pytest

from compitum.cli import _load_constraints, _toy_models
from compitum.boundary import BoundaryAnalyzer
from compitum.coherence import CoherenceFunctional
from compitum.constraints import ReflectiveConstraintSolver
from compitum.control import LyapunovController
from compitum.energy import SymbolicFreeEnergy
from compitum.metric import SymbolicManifoldMetric
from compitum.pgd import RegexPromptExtractor
from compitum.predictors import CalibratedPredictor
from compitum.router import CompitumRouter
from pathlib import Path
import numpy as np
import yaml


def build_router(seed: int = 12345) -> CompitumRouter:
    dcfg = yaml.safe_load(Path("configs/router_defaults.yaml").read_text())
    D, rank, delta = int(dcfg["metric"]["D"]), int(dcfg["metric"]["rank"]), float(dcfg["metric"]["delta"])
    models = _toy_models(D)
    rng = np.random.default_rng(seed)
    X_demo = rng.standard_normal((512, D))
    predictors = {}
    for m in models:
        yq = 0.6 + 0.1 * np.tanh(X_demo @ (m.center / np.linalg.norm(m.center) + 1e-8))
        yt = 0.5 + 0.5 * np.abs(X_demo @ np.ones(D) / np.sqrt(D))
        yc = 0.2 + 0.4 * np.abs(X_demo @ (np.arange(D) / D))
        pq = CalibratedPredictor(); pq.fit(X_demo, yq)
        pt = CalibratedPredictor(); pt.fit(X_demo, yt)
        pc = CalibratedPredictor(); pc.fit(X_demo, yc)
        predictors[m.name] = {"quality": pq, "latency": pt, "cost": pc}
    metrics = {m.name: SymbolicManifoldMetric(D, rank, delta) for m in models}
    coherence = CoherenceFunctional(k=500)
    A, b = _load_constraints(Path("configs/constraints_us_default.yaml"))
    solver = ReflectiveConstraintSolver(A, b)
    boundary = BoundaryAnalyzer()
    controller = LyapunovController()
    energy = SymbolicFreeEnergy(dcfg["alpha"], dcfg["beta_t"], dcfg["beta_c"], dcfg["beta_d"], dcfg["beta_s"])
    return CompitumRouter(
        models, predictors, solver, coherence, boundary, controller,
        RegexPromptExtractor(), metrics, energy, update_stride=int(dcfg["update_stride"]),
    )


@pytest.fixture
def router():
    return build_router()


def test_smoke_basic_route(router):
    cert = router.route("Test prompt")
    assert cert.constraints["feasible"], "Route infeasible"


def test_smoke_constraint_enforcement(router):
    cert = router.route("Test prompt")
    # shadow_prices is a dict keyed by constraint row (lambda_0, lambda_1, ...)
    for name, value in cert.constraints["shadow_prices"].items():
        assert value >= 0.0, f"Unexpected negative shadow price: {name}"


@pytest.mark.parametrize("prompt", [
    "Simple question",
    "Complex multi-part query with context",
    "Short",
])
def test_smoke_prompt_variety(router, prompt):
    cert = router.route(prompt)
    assert cert.constraints["feasible"]
